Short Term Memory

A basic feature, prominently missing from the agents we have been creating so far, is the ability to maintain memory over the lebnth of a conversation.
Let's test this out.

In [1]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage

model = ChatGoogleGenerativeAI(model= "gemini-2.5-flash-lite")

agent =create_agent(model= model)

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")

response = agent.invoke(
    {"messages": [question]} 
)

In [2]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='22173fc6-5e23-46f9-85bf-c31c3f87f4e9'),
              AIMessage(content="Hello Seán! It's nice to meet you. Green is a wonderful favorite color. It's often associated with nature, growth, and tranquility. Do you have a particular shade of green you like best, or is it green in general?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd671-fbac-76d0-a8a0-008093f88e94-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 51, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}})]}


If we run the same agent again...

In [3]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]} 
)

pprint(response)

{'messages': [HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='21a40c82-6a9e-4056-8378-5872930f0161'),
              AIMessage(content="I cannot know your favorite color. As a large language model, I do not have access to personal information about you, including your preferences.\n\nTo tell me your favorite color, you'll have to tell me! 😊", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd673-e9d0-7040-bcc6-97c76e44f28c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 45, 'total_tokens': 53, 'input_token_details': {'cache_read': 0}})]}


It has forgotten all about our previous interaction.

What has happend here?

With our langchain agent, we are tracking messages with something called the state, which can be thought of as the memory of our agent.
Problem is, the state isn't being saved each time the agent is being run.

We can "save the state" using a pointer, which takes a snapshot at the end of each agent run and groups it into an id.

We use the langgraph library 'in mmemory saver'

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  #import the library InMemorySaver


agent = create_agent(
    model= model,
    checkpointer=InMemorySaver(),  
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Hello my name is Seán and my favourite colour is green")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [6]:
pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='d5571b80-9036-483d-a93a-2fc91ad125a2'),
              AIMessage(content="Hello Seán! It's nice to meet you. Green is a wonderful color! It's associated with nature, growth, and tranquility. Do you have a favorite shade of green, or anything in particular that makes you like it so much?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd67a-d3ab-7293-9411-26fbb5e769f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 51, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}})]}


In [7]:
question = HumanMessage(content="What's my favourite colour?")

response = agent.invoke(
    {"messages": [question]},
    config,  
)

pprint(response)

{'messages': [HumanMessage(content='Hello my name is Seán and my favourite colour is green', additional_kwargs={}, response_metadata={}, id='d5571b80-9036-483d-a93a-2fc91ad125a2'),
              AIMessage(content="Hello Seán! It's nice to meet you. Green is a wonderful color! It's associated with nature, growth, and tranquility. Do you have a favorite shade of green, or anything in particular that makes you like it so much?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bd67a-d3ab-7293-9411-26fbb5e769f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 51, 'total_tokens': 64, 'input_token_details': {'cache_read': 0}}),
              HumanMessage(content="What's my favourite colour?", additional_kwargs={}, response_metadata={}, id='c0610791-6358-4d74-9254-41e148fa92e8'),
              AIMessage(content='You menti

By default, our agent state tracks only a list of messages. This can be customized however to track fields like user id for example, and language, if we'd like that to be tracked over time.

This will be discussed further in the next chapter.